# Retail geoespacial: ¿dos tiendas atienden a los mismos clientes?

**Moda Punto** tiene dos tiendas ficticias y cinco zonas de clientes. La gerencia quiere saber
qué zonas están cubiertas por ambas tiendas, cuáles por una sola y cuáles quedan fuera de las dos.
Esto permite detectar posible canibalización y oportunidades de cobertura.

Los datos son inventados. Usaremos un radio de atención de 5 km en línea recta. En Colab, suba
este archivo y seleccione **Entorno de ejecución → Ejecutar todas**.

## ¿Qué técnica estamos usando?

El análisis geoespacial combina una tabla con la ubicación de cada registro. Aquí cada tienda y
cada zona de clientes se representa como un **punto**. La longitud indica la posición este-oeste
y la latitud la posición norte-sur.

Primero guardamos esos puntos en GeoPandas. Después los transformamos a un sistema de coordenadas
que trabaja en metros. Esto es necesario porque las coordenadas originales están expresadas en
grados: no sería correcto restar dos longitudes y llamar kilómetros al resultado.

Luego usamos un **buffer**, que es un área dibujada alrededor de un punto con una distancia fija.
Un buffer de 5 km alrededor de una tienda representa una regla sencilla de atención: consideramos
cercana una zona cuyo punto cae dentro de ese círculo. Cuando los dos círculos se superponen,
aparece una zona compartida. Esa zona compartida señala posible canibalización, porque ambas tiendas
podrían competir por la misma demanda.

La técnica ayuda a ordenar preguntas de negocio: dónde hay cobertura, dónde se repite la cobertura
y dónde existe demanda fuera de las áreas actuales. Es una primera aproximación, no una respuesta
definitiva. Un cliente puede viajar más de 5 km, una carretera puede hacer que dos puntos cercanos
tarden mucho en conectarse y una zona sin cobertura puede no tener suficiente demanda para sostener
una tienda.

## 1. Preparar herramientas

GeoPandas agrega geometrías a tablas de pandas. Matplotlib dibuja el resultado.

In [ ]:
%pip install -q "geopandas==1.1.4" "matplotlib>=3.8,<4" "pandas>=2.2,<3" "numpy>=1.26,<3"

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from pathlib import Path
SALIDA = Path("salidas_cannibalizacion_retail")
SALIDA.mkdir(exist_ok=True)

## 2. Crear los datos

Cada tienda y cada zona se representa con longitud y latitud. `clientes_mes` indica demanda
mensual simulada de una zona; no son personas identificadas.

In [ ]:
tiendas_df = pd.DataFrame({
    "tienda": ["Moda Punto Norte", "Moda Punto Sur"],
    "longitud": [-100.30, -100.30], "latitud": [25.78, 25.62],
})
zonas_df = pd.DataFrame({
    "zona": ["Centro", "Noreste", "Sureste", "Suroeste", "Oeste"],
    "longitud": [-100.30, -100.20, -100.18, -100.43, -100.48],
    "latitud": [25.70, 25.82, 25.57, 25.60, 25.72],
    "clientes_mes": [620, 310, 260, 180, 450],
})
display(tiendas_df); display(zonas_df)

La zona Centro concentra más clientes que la zona Suroeste. Por eso, al resumir, veremos
tanto el número de zonas como el número de clientes.

## 3. Convertir las tablas en puntos y medir distancias

Primero usamos EPSG:4326 para ubicar los puntos. Después usamos EPSG:32614, donde las unidades
son metros, para medir correctamente. Una zona está “cerca” si está a 5 km o menos.

In [ ]:
tiendas = gpd.GeoDataFrame(tiendas_df, geometry=gpd.points_from_xy(tiendas_df.longitud, tiendas_df.latitud), crs="EPSG:4326")
zonas = gpd.GeoDataFrame(zonas_df, geometry=gpd.points_from_xy(zonas_df.longitud, zonas_df.latitud), crs="EPSG:4326")
tiendas_m = tiendas.to_crs("EPSG:32614")
zonas_m = zonas.to_crs("EPSG:32614")
for fila in tiendas_m.itertuples():
    sufijo = "norte" if "Norte" in fila.tienda else "sur"
    zonas_m[f"dist_{sufijo}_km"] = zonas_m.geometry.distance(fila.geometry) / 1000
zonas_m["cerca_norte"] = zonas_m.dist_norte_km <= 5
zonas_m["cerca_sur"] = zonas_m.dist_sur_km <= 5
zonas_m["cobertura"] = np.select(
    [zonas_m.cerca_norte & zonas_m.cerca_sur, zonas_m.cerca_norte, zonas_m.cerca_sur],
    ["Ambas tiendas", "Solo Norte", "Solo Sur"], default="Ninguna tienda")
resultado = zonas_m[["zona", "clientes_mes", "dist_norte_km", "dist_sur_km", "cobertura"]].sort_values("zona")
display(resultado)

La tabla indica la distancia a cada tienda y resume la cobertura. Las zonas “Ambas tiendas”
son las que pueden tener solapamiento de clientes. “Ninguna tienda” identifica una oportunidad
para estudiar, pero no demuestra por sí sola que abrir una tienda sea rentable.

## 4. Resumir canibalización potencial y oportunidad

Agrupamos por `cobertura`. `sum` suma los clientes de cada grupo. El porcentaje usa todos los
clientes como denominador, por lo que refleja la importancia comercial, no solo el número de zonas.

In [ ]:
resumen = resultado.groupby("cobertura", as_index=False).agg(
    zonas=("zona", "size"), clientes_mes=("clientes_mes", "sum"))
resumen["porcentaje_clientes"] = 100 * resumen.clientes_mes / zonas_df.clientes_mes.sum()
orden = {"Ambas tiendas": 0, "Solo Norte": 1, "Solo Sur": 2, "Ninguna tienda": 3}
resumen["orden"] = resumen.cobertura.map(orden)
resumen = resumen.sort_values("orden").drop(columns="orden")
compartidos = int(resumen.loc[resumen.cobertura == "Ambas tiendas", "clientes_mes"].sum())
sin_cobertura = int(resumen.loc[resumen.cobertura == "Ninguna tienda", "clientes_mes"].sum())
total = int(zonas_df.clientes_mes.sum())
display(resumen)
display(Markdown(f"En este caso hay **{compartidos} clientes mensuales** en zonas cubiertas por ambas tiendas y **{sin_cobertura} clientes** fuera de las dos. Son señales para investigar, no ventas perdidas o ventas nuevas confirmadas."))

## 5. Dibujar los radios de atención

`buffer(5000)` crea un círculo de 5,000 metros alrededor de cada tienda. Los colores de las zonas
se basan en la clasificación calculada anteriormente.

In [ ]:
radios = tiendas_m.copy(); radios["geometry"] = radios.geometry.buffer(5000); radios = radios.to_crs("EPSG:4326")
zonas_mapa = zonas_m.to_crs("EPSG:4326")
colores = {"Ambas tiendas":"#c83f49", "Solo Norte":"#e39b32", "Solo Sur":"#7a62a8", "Ninguna tienda":"#4c9a65"}
colores_tiendas = {"Moda Punto Norte": "#1769aa", "Moda Punto Sur": "#8e44ad"}
fig, ax = plt.subplots(figsize=(9, 6))
for f in radios.itertuples():
    gpd.GeoSeries([f.geometry], crs="EPSG:4326").plot(ax=ax, color=colores_tiendas[f.tienda], alpha=.14,
        edgecolor=colores_tiendas[f.tienda], label=f"Radio de {f.tienda}")
for categoria, color in colores.items():
    subset = zonas_mapa[zonas_mapa.cobertura == categoria]
    if not subset.empty:
        subset.plot(ax=ax, color=color, markersize=120, label=categoria)
for f in tiendas.itertuples():
    gpd.GeoSeries([f.geometry], crs="EPSG:4326").plot(ax=ax, color=colores_tiendas[f.tienda], marker="*", markersize=280, label=f.tienda)
for f in zonas_mapa.itertuples(): ax.annotate(f"{f.zona} ({int(f.clientes_mes)})", (f.geometry.x, f.geometry.y), xytext=(5,5), textcoords="offset points", fontsize=8)
ax.set(title="Moda Punto: cobertura y zonas compartidas", xlabel="Longitud", ylabel="Latitud"); ax.legend(fontsize=8); ax.grid(alpha=.2); fig.tight_layout(); fig.savefig(SALIDA / "mapa_cannibalizacion.png", bbox_inches="tight"); plt.show()

Los puntos rojos están dentro del radio de ambas tiendas: son las zonas prioritarias para
revisar posible canibalización. Los puntos verdes están fuera de las dos áreas: son oportunidades
para investigar, no una decisión automática de apertura.

## Explicación detallada de cada bloque de código

Esta sección conserva los bloques anteriores y explica con más calma qué ocurre en cada uno.

### Bloque 1: instalar las librerías

`%pip install` le dice a Colab que descargue las herramientas necesarias. `-q` significa “quieto”:
reduce mensajes de instalación para que podamos concentrarnos en el análisis. `geopandas==1.1.4`
fija una versión concreta; así el resultado no depende de cambios futuros. Los símbolos `>=` y `<`
de las otras librerías indican un intervalo de versiones compatibles. Este bloque no analiza tiendas,
solo prepara el espacio de trabajo. Si las librerías ya están instaladas, Colab normalmente termina rápido.

### Bloque 2: importar y preparar el espacio de salida

`import numpy as np` carga herramientas numéricas; `pandas as pd` carga tablas; `geopandas as gpd`
carga tablas que también conocen geometrías; y `matplotlib.pyplot as plt` sirve para dibujar.
`display` muestra tablas y texto con formato dentro del notebook. `Path` crea rutas que funcionan
en distintos sistemas. `SALIDA` es una carpeta para guardar archivos y `mkdir(exist_ok=True)` la
crea sin producir error si ya existe. La opción `display.float_format` solo cambia cómo se ven
los decimales; no cambia los valores guardados.

### Bloque 3: crear los datos ficticios

`pd.DataFrame({...})` construye una tabla: cada clave entre comillas es una columna y cada lista
aporta una fila. En `tiendas_df`, cada fila representa una tienda. En `zonas_df`, cada fila representa
una zona, y `clientes_mes` es una cantidad simulada. Los nombres de las columnas permiten leer el
código como una hoja de cálculo. `display` enseña las tablas para revisar que las filas y los números
sean razonables antes de crear el mapa.

### Bloque 4: convertir tablas en puntos y dibujar el mapa inicial

`gpd.GeoDataFrame` mantiene las columnas de la tabla y añade una columna `geometry`. La función
`points_from_xy` combina longitud y latitud para crear un punto; la longitud va primero porque
representa el eje horizontal. `crs="EPSG:4326"` declara que esas coordenadas están en grados.
Las expresiones entre corchetes, como `lugares[lugares.tipo == ...]`, filtran filas: dejan solo
tiendas o solo zonas. `.plot` dibuja cada grupo; `marker="*"` cambia el símbolo de las tiendas.
El ciclo `for` visita cada fila para escribir su etiqueta. `legend` explica colores y símbolos,
`grid` agrega una cuadrícula, `tight_layout` acomoda el espacio y `savefig` guarda una imagen.

### Bloque 5: proyectar y medir distancias

`to_crs("EPSG:32614")` transforma los puntos a un sistema local cuyas unidades son metros.
`itertuples()` recorre las dos tiendas una por una. Para cada tienda, el `if` elige el texto
`norte` o `sur`, y se crea una columna de distancia. `geometry.distance(...)` mide cada zona desde
esa tienda; dividir entre 1,000 convierte metros a kilómetros. Las comparaciones `<= 5` producen
valores verdadero/falso. `np.select` convierte la combinación de esos valores en una etiqueta:
si ambas son verdaderas, la zona es compartida; si solo una lo es, pertenece a una tienda; si
ninguna lo es, queda sin cobertura. La tabla `resultado` selecciona columnas útiles y ordena las zonas.

### Bloque 6: resumir la demanda

`groupby("cobertura")` reúne las zonas que tienen la misma situación. `agg` cuenta zonas con `size`
y suma clientes con `sum`. El porcentaje divide los clientes de cada grupo entre el total de clientes.
El diccionario `orden` solo fija un orden de lectura. `loc` filtra el grupo compartido o el grupo
sin cobertura, y `sum` obtiene su demanda. `int` elimina decimales porque estamos contando clientes.
La f-string `f"...{valor}..."` inserta los resultados en un texto explicativo. Este bloque convierte
varios puntos del mapa en una medida comercial que un gerente puede discutir.

### Bloque 7: construir los radios y el mapa final

`copy` conserva las tiendas originales antes de modificar una copia. `buffer(5000)` dibuja un área
de 5,000 metros alrededor de cada tienda. `to_crs("EPSG:4326")` devuelve las geometrías a grados
para graficarlas con las tablas originales. El diccionario `colores` asigna un color a cada categoría
de cobertura y `colores_tiendas` asigna azul a Norte y morado a Sur. El ciclo de categorías filtra
una categoría y solo la dibuja si tiene filas; esto evita errores cuando alguna categoría está vacía.
Los ciclos siguientes dibujan cada radio, cada zona y las etiquetas. `savefig` guarda el mapa para
consultarlo sin volver a ejecutar el notebook.

### Bloque 8: mostrar conclusiones y guardar resultados

La expresión dentro de `Markdown` vuelve a mostrar los conteos principales en lenguaje de negocio.
`to_csv` guarda la clasificación y el resumen en archivos que se pueden abrir en Excel. Estos archivos
permiten revisar el resultado sin depender únicamente de la imagen. La instrucción `print` confirma
la carpeta donde se guardaron. Guardar resultados es importante porque el mapa explica visualmente,
pero las tablas permiten auditar qué zona produjo cada total.

### Cómo revisar el código como principiante

En cada bloque siga este orden: primero identifique la tabla que entra, después observe la columna
que se crea y finalmente revise qué pregunta responde la salida. Los nombres con `_df` son tablas
normales; los nombres sin ese sufijo, como `zonas_m`, son tablas geográficas o transformadas.
Las variables `cerca_norte` y `cerca_sur` son respuestas de sí/no. La columna `cobertura` traduce
esas dos respuestas a una frase que se puede agrupar y contar.

## 6. Conclusiones

Moda Punto puede separar la demanda en cuatro grupos: compartida, exclusiva de Norte, exclusiva
de Sur y sin cobertura. Este análisis es una primera pantalla territorial. Para tomar una decisión
real habría que añadir ventas por tienda, tiempo de viaje, competencia, renta y costos de apertura.

In [ ]:
display(Markdown(f"""
### Resumen ejecutivo
- Zonas compartidas: **{int(resumen.loc[resumen.cobertura == 'Ambas tiendas', 'zonas'].sum())}**, con **{compartidos} clientes mensuales**.
- Zonas sin cobertura de 5 km: **{int(resumen.loc[resumen.cobertura == 'Ninguna tienda', 'zonas'].sum())}**, con **{sin_cobertura} clientes mensuales**.
- Siguiente paso: estudiar las zonas rojas para solapamiento y las verdes para una posible expansión.
"""))
resultado.to_csv(SALIDA / "clasificacion_zonas.csv", index=False)
resumen.to_csv(SALIDA / "resumen_cobertura.csv", index=False)
print("Resultados guardados en", SALIDA.resolve())